In [0]:
import pandas as pd                                                                 # Import pandas for data cleaning
import numpy as np                                                                  # Import Numpy for Maths functions
import matplotlib.pyplot as plt                                                     # Import Matplot for Viz functions
import seaborn as sns                                                               # Import Seaborn for visualization
import matplotlib.pyplot as plt                                                     # Import matplotlib library for visualization
import plotly.express as px                                                         # Import plotly library for visualization
import plotly.graph_objects as go

from sklearn.ensemble import IsolationForest                                        # EDA-Isolation Forest Analysis Functions

from pyspark.sql import functions as F
from pyspark.sql.functions import col, StringType, NumericType                      # TableFunctions
from pyspark.sql.functions import mean, min, max, stddev, count, sum as _sum        # MathsFunctions
from pyspark.sql.functions import to_date, year, month, datediff                    # DateFunctions
from pyspark.sql.functions import abs                                               # OtherFunctions

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

from sklearn.linear_model import LinearRegression                                   # LinearRegression Analysis Functions
from sklearn.metrics import r2_score, mean_squared_error                            # LinearRegression Analysis Functions

from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder        # Classification Analysis Functions
from pyspark.ml import Pipeline                                                     # Classification Analysis Functions
from sklearn.linear_model import LogisticRegression                                 # Classification Analysis Functions
from sklearn.metrics import mean_squared_error, r2_score                            # Classification Analysis Functions
from sklearn.impute import SimpleImputer                                            # Classification Analysis Functions
from sklearn.preprocessing import LabelEncoder                                      # Classification Analysis Functions
from sklearn.preprocessing import OneHotEncoder                                     # Classification Analysis Functions
from sklearn.preprocessing import StandardScaler                                    # Classification Analysis Functions
from sklearn.model_selection import train_test_split                                # Classification Analysis Functions
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report # Classification Analysis Functions
from sklearn.metrics import roc_curve, auc                                          # Classification Analysis Functions

from sklearn.tree import DecisionTreeClassifier                                     # DecisionTree Analysis Functions

from sklearn.ensemble import RandomForestClassifier                                 # RandomForest Analysis Functions

from sklearn.cluster import KMeans                                                  # KMeans Cluster Analysis Functions


In [0]:
# Load dataset as Spark DataFrame

df = spark.table("lifeexpectancy.brone_lifeexpectancy.life_expectancy")
df_raw = df

# display(df)

In [0]:
from pyspark.sql import functions as F

# Load Bronze table
df = spark.table("lifeexpectancy.brone_lifeexpectancy.life_expectancy")

# Cast DecimalType → double
for col, dtype in df.dtypes:
    if "decimal" in dtype:
        df = df.withColumn(col, F.col(col).cast("double"))

# Numeric columns → median fill
numeric_cols = [c for (c, t) in df.dtypes if t in ("int", "double", "float")]
for col in numeric_cols:
    median_val = df.approxQuantile(col, [0.5], 0.01)[0]
    df = df.withColumn(col, F.when(F.col(col).isNull(), median_val).otherwise(F.col(col)))

# Categorical columns → mode fill
categorical_cols = [c for (c, t) in df.dtypes if t == "string"]
for col in categorical_cols:
    mode_val = df.groupBy(col).count().orderBy(F.desc("count")).first()[0]
    df = df.withColumn(col, F.when(F.col(col).isNull(), mode_val).otherwise(F.col(col)))

# Drop old Silver table to avoid schema conflicts
spark.sql("DROP TABLE IF EXISTS lifeexpectancy.silver.life_expectancy")

# Save cleaned data to Silver
df.write.option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("lifeexpectancy.silver.life_expectancy")


In [0]:
# HANDLING NULLS
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
])
null_counts.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("lifeexpectancy.silver.lifeexpectancy_nulls")

In [0]:
# Numeric columns to check
numeric_cols = [c for (c, t) in df.dtypes if t in ("int", "double", "float")]

# Function to add outlier flag for a column
def add_outlier_flag(df, col):
    q1, q3 = df.approxQuantile(col, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    return df.withColumn(
        f"{col}_Outlier",
        F.when((F.col(col) < lower_bound) | (F.col(col) > upper_bound), 1).otherwise(0)
    )

# Apply outlier flagging for each numeric column
for col_name in numeric_cols:
    df = add_outlier_flag(df, col_name)

# Show sample with flags
df.select(numeric_cols + [f"{c}_Outlier" for c in numeric_cols]).show(10)

# Force overwrite with schema alignment
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "lifeexpectancy.silver.lifeexpectancy_outliners")

In [0]:
from pyspark.sql import functions as F
from sklearn.ensemble import IsolationForest
import pandas as pd

# Step 1: Load Spark DataFrame (Bronze/Silver source)
df = spark.table("lifeexpectancy.silver.life_expectancy")

# Step 2: Cast all DecimalType columns to double
for col, dtype in df.dtypes:
    if "decimal" in dtype:
        df = df.withColumn(col, F.col(col).cast("double"))

# Step 3: Select numeric columns
numeric_cols = [
    "Year","Adult_Mortality","infant_deaths","Alcohol","percentage_expenditure",
    "Hepatitis_B","Measles","BMI","under_five_deaths","Polio","Total_expenditure",
    "Diphtheria","HIV_AIDS","GDP","Population","thinness_1_19_years",
    "thinness_5_9_years","Income_composition_of_resources","Schooling","Life_expectancy"
]

# Step 4: Convert to Pandas for sklearn
pdf = df.select(numeric_cols).toPandas()

# Step 5: Fill missing values (median for numeric)
for col in numeric_cols:
    pdf[col] = pdf[col].fillna(pdf[col].median())

# Step 6: Fit Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
pdf["anomaly_pred"] = iso.fit_predict(pdf[numeric_cols])   # -1 = anomaly, 1 = normal
pdf["anomaly_score"] = iso.decision_function(pdf[numeric_cols])

# Step 7: Convert back to Spark
df_anomaly = spark.createDataFrame(pdf)

# Step 8: Drop old Silver table to avoid schema conflicts
spark.sql("DROP TABLE IF EXISTS lifeexpectancy.silver.lifeexpectancy_iforest")

# Step 9: Save anomaly results to Silver table with overwriteSchema
df_anomaly.write.option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("lifeexpectancy.silver.lifeexpectancy_iforest")
